# Benchmark summary tables

Scores all methods and draws one table per cohort plus a combined table.

- Metrics are averaged over samples, then min–max scaled per cohort across non-control methods.
- MECR is size-adjusted per cohort and inverted. Cell density is log-scaled.
- The combined table averages scaled metrics across cohorts.
- The total score is the mean of the available categories.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
%pip install -q rpy2 "setuptools<81"
!mamba install -y -q -p {sys.prefix} -c conda-forge r-svglite r-showtext r-dplyr r-tibble r-ggplot2 r-rcolorbrewer r-ggforce r-jsonlite

In [2]:
%load_ext rpy2.ipython

In [3]:
%%R
if (!requireNamespace("funkyheatmap", quietly = TRUE))
  install.packages("funkyheatmap", repos = "https://cloud.r-project.org")

In [4]:
sys.path.insert(0, "/dss/dsshome1/0C/ra98gaq/Git/branch-metrics/cellseg-benchmark")
import cellseg_benchmark._constants as _constants
from cellseg_benchmark._constants import main_methods
from cellseg_benchmark.metrics import clean_method_name

main_dir = Path.cwd().parents[0] / "data"
out_dir = main_dir / "metrics" / "merged" / "benchmark_tables"
out_dir.mkdir(parents=True, exist_ok=True)

cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
exclude = ["Negative_Control_Visium", "Ficture_segments"]
output_folders = ["general_stats", "assigned_transcripts", "cell_type_metrics", "marker_gene_metrics", "ficture", "cell_density"]
metric_tables = [
    "general_stats", "assigned_transcript_counts", "cell_density", "MECR_score_all", "ficture_f1",
    "marker_f1_score_cell_type_revised", "clustering_score_adata_integrated_cell_type_revised",
]

categories = {
    "clustering": ["silhouette_score"],
    "marker": ["MECR_adj", "marker_f1"],
    "ficture": ["ficture_f1_score"],
    "z": ["Ovrlpy_stats_mean_integrity"],
}
scored = sum(categories.values(), [])
descriptive = ["volume_final", "elongation", "intensities_DAPI", "cells_per_mm2", "pct_assigned_qced"]
metrics = scored + descriptive
metric_labels = pd.DataFrame(list(_constants.metric_names.items()), columns=["id", "name"])

/home/ubuntu/miniforge3/envs/analysis/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/ubuntu/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load metrics

In [5]:
dfs_by = {
    cohort: {
        csv.stem: pd.read_csv(csv, index_col=0).query("sample not in ['all', 'all_samples']")
        for folder in output_folders
        for csv in (main_dir / "metrics" / cohort / folder).glob("*.csv")
        if csv.stem in metric_tables
    }
    for cohort in cohorts
}
dfs_by["combined"] = {table: pd.concat([dfs_by[c][table] for c in cohorts], ignore_index=True) for table in metric_tables}

## Build tables

In [6]:
def add_scores(df):
    category_scores = pd.DataFrame({name: df[cols].mean(axis=1) for name, cols in categories.items()})
    df["total_score"] = category_scores.mean(axis=1)
    df["expr_score"] = category_scores[["clustering", "marker"]].mean(axis=1)
    return df.sort_values("total_score", ascending=False)


def build_benchmark_table(dfs):
    general = dfs["general_stats"].query("cell_type_revised == 'all'")
    counts = dfs["assigned_transcript_counts"].groupby(["sample", "method"])[["assigned_count_qced", "total_count"]].sum()
    counts["pct_assigned_qced"] = counts.assigned_count_qced / counts.total_count
    counts["tpc"] = counts.assigned_count_qced / general.set_index(["sample", "method"]).n_cells
    density = dfs["cell_density"].groupby("method")[["n_cells", "tissue_mm2"]].sum()
    marker_f1 = (
        dfs["marker_f1_score_cell_type_revised"]
        .groupby(["sample", "method", "cell_type"]).f1_score.mean()
        .groupby(["sample", "method"]).mean()
        .groupby("method").mean()
    )

    df = pd.concat([
        general.groupby("method")[["volume_final", "elongation", "intensities_DAPI", "Ovrlpy_stats_mean_integrity"]].mean(),
        counts.groupby("method")[["pct_assigned_qced", "tpc"]].mean(),
        dfs["clustering_score_adata_integrated_cell_type_revised"].groupby("method")[["silhouette_score"]].mean(),
        dfs["MECR_score_all"].query("gene1 == 'all'").groupby("method")[["MECR"]].mean(),
        marker_f1.rename("marker_f1"),
        dfs["ficture_f1"].query("cell_type == 'f1_macro_all'").groupby("method").f1.mean().rename("ficture_f1_score"),
        (density.n_cells / density.tissue_mm2).rename("cells_per_mm2"),
    ], axis=1).drop(index=exclude, errors="ignore")

    df["is_control"] = df.index.str.startswith("Negative_Control")
    real = df[~df.is_control]
    fit_data = real.dropna(subset=["MECR", "tpc"])
    fit = np.polyfit(np.log10(fit_data.tpc), fit_data.MECR, 1)
    df["MECR_adj"] = df.MECR - np.polyval(fit, np.log10(df.tpc))

    is_3d = df.index.str.startswith(tuple(_constants.methods_3D))
    is_ficture_seg = df.index.str.startswith("Ficture_segments")
    df.loc[is_ficture_seg, "ficture_f1_score"] = np.nan
    df["na_z"] = is_3d & df.Ovrlpy_stats_mean_integrity.isna()
    df["na_ficture"] = (is_3d | is_ficture_seg) & df.ficture_f1_score.isna()
    df["density_label"] = (df.cells_per_mm2 / 1e3).map("{:.1f}".format)

    values = df[metrics].assign(MECR_adj=-df.MECR_adj, cells_per_mm2=np.log10(df.cells_per_mm2))
    low, high = values[~df.is_control].min(), values[~df.is_control].max()
    df[metrics] = ((values - low) / (high - low)).clip(0, 1)

    df.index = df.index.map(clean_method_name).rename("id")
    return add_scores(df)


tables = {name: build_benchmark_table(dfs) for name, dfs in dfs_by.items()}

combined = tables["combined"]
pooled_score = combined.total_score.copy()
combined[metrics] = pd.concat([tables[c][metrics] for c in cohorts]).groupby(level=0).mean().reindex(combined.index)
tables["combined"] = add_scores(combined)

## Plot tables

In [7]:
%%R
plot_benchmark_heatmap <- function(df_funky, out_file, metric_labels, family = "Arial", font_file = "../data/misc/Arial.ttf", score_label = "Overall") {
  library(funkyheatmap)
  library(dplyr)
  library(tibble)
  library(ggplot2)
  library(RColorBrewer)

  sysfonts::font_add(family, path.expand(font_file))
  showtext::showtext_auto()
  showtext::showtext_opts(dpi = 600)
  update_geom_defaults("text", list(family = family))
  for (s in c("sep1", "sep2", "sep3")) df_funky[[s]] <- ""

  box <- list(width = 1.02)
  col_spec <- tribble(
    ~id,                          ~group,       ~geom,       ~palette,      ~name,          ~opts,
    "id",                         "",           "text",      NA,            "",             list(width = 8.8, hjust = 0),
    "total_score",                "overall",    "bar",       "overall",     "",             list(width = 3.2, draw_outline = FALSE),
    "sep1",                       "",           "text",      NA,            "",             list(width = 0.3),
    "silhouette_score",           "expression", "funkyrect", "expression",  "Silhouette",   box,
    "MECR_adj",                   "expression", "funkyrect", "expression",  "MECR",         box,
    "marker_f1",                  "expression", "funkyrect", "expression",  "Marker F1",    box,
    "sep2",                       "",           "text",      NA,            "",             list(width = 0.3),
    "ficture_f1_score",           "spatial",    "funkyrect", "spatial",     "Ficture F1",   box,
    "Ovrlpy_stats_mean_integrity","spatial",    "funkyrect", "spatial",     "Z integrity",  box,
    "sep3",                       "",           "text",      NA,            "",             list(width = 0.8),
    "pct_assigned_qced",          "general",    "funkyrect", "transcripts", "% Assigned",   box,
    "volume_final",               "general",    "funkyrect", "general",     "Volume",       box,
    "elongation",                 "general",    "funkyrect", "general",     "Elongation",   box,
    "intensities_DAPI",           "general",    "funkyrect", "general",     "DAPI",         box,
    "cells_per_mm2",              "general",    "rect",      "general_lt",  "Cells/mm²",    list(width = 1.1),
    "density_label",              "general",    "text",      NA,            "",             list(overlay = TRUE, size = 2.4, hjust = 0.5),
  )

  column_info <- col_spec %>%
    left_join(metric_labels, by = "id", suffix = c("", "_label")) %>%
    mutate(name = coalesce(name_label, name), name = ifelse(id == "cells_per_mm2", "Cells/mm² (×10³)", name),
           options = opts, legend = FALSE) %>%
    select(-name_label, -opts)

  column_groups <- tribble(
    ~group,       ~level1,      ~palette,
    "overall",    score_label,  "overall",
    "expression", "Expression", "expression",
    "spatial",    "Spatial",    "spatial",
    "general",    "General (unscored)", "general"
  )

  pal <- function(name, i = 2:7) brewer.pal(9, name)[i]
  palettes <- list(
    overall     = colorRampPalette(c("#FFF3CF", "#FBC36A", "#E08A3C", "#C46A35"))(6),
    expression  = pal("Purples"),
    spatial     = colorRampPalette(c("#FCE4DC", "#C25B4E"))(6),
    transcripts = pal("Blues"),
    general     = pal("Greens"),
    general_lt  = pal("Greens", 1:4)
  )

  legends <- list(
    list(title = if (score_label == "Overall") "Total score" else score_label, palette = "overall", geom = "bar", labels = c("0", "", "0.5", "", "1"),
         size = c(0, 0.25, 0.5, 0.75, 1), color = palettes$overall[c(1, 2, 4, 5, 6)]),
    list(title = "Scaled metric", palette = "expression", geom = "funkyrect",
         labels = c("min", "", "", "", "max"), size = c(0.1, 0.3, 0.5, 0.75, 1),
         color = pal("Greys", c(2, 3, 5, 6, 7)), label_hjust = 0),
    list(palette = "spatial",     enabled = FALSE),
    list(palette = "transcripts", enabled = FALSE),
    list(palette = "general",     enabled = FALSE),
    list(palette = "general_lt",  enabled = FALSE)
  )

  g <- funky_heatmap(
    df_funky,
    column_info   = column_info,
    column_groups = column_groups,
    palettes      = palettes,
    legends       = legends,
    scale_column  = FALSE,
    position_args = position_arguments(
      row_height = 1.02, row_space = 0.08, col_space = 0.08,
      expand_xmax = 4.5, expand_ymin = 0,
      expand_ymax = 0.3, col_annot_offset = 3.6, col_annot_angle = 45
    )
  )

  p <- g[[1]]
  rl <- which(sapply(p$layers, function(l) inherits(l$geom, "GeomRect") && "column_id" %in% names(l$data)))
  p$layers[[rl]]$data <- mutate(p$layers[[rl]]$data, d = ifelse(column_id %in% "cells_per_mm2", 0.08, 0),
                                xmin = xmin + d, xmax = xmax - d, ymin = ymin + d, ymax = ymax - d)
  pos <- bind_rows(lapply(p$layers, function(l) mutate(l$data[intersect(names(l$data), c("label_value", "row_id", "xmin", "xmax", "ymin", "ymax"))], across(everything(), as.character)))) %>%
    mutate(across(c(xmin, xmax, ymin, ymax), as.numeric))
  col_x <- function(i) with(filter(pos, label_value == column_info$name[column_info$id == i])[1, ], c(xmin, xmax))
  rows <- pos %>% filter(row_id %in% df_funky$id) %>% group_by(row_id) %>% summarise(y = (min(ymin) + max(ymax)) / 2)
  z <- col_x("Ovrlpy_stats_mean_integrity")
  na <- bind_rows(tibble(row_id = df_funky$id[df_funky$na_ficture], x = mean(col_x("ficture_f1_score"))),
                  tibble(row_id = df_funky$id[df_funky$na_z], x = mean(z))) %>% left_join(rows, by = "row_id")
  hdr <- which(sapply(p$layers, function(l) inherits(l$geom, "GeomText")))
  p$layers[[hdr]]$data <- mutate(p$layers[[hdr]]$data, size = ifelse(label_value %in% column_groups$level1, 3.8, size))
  bar <- with(filter(pos, label_value == score_label)[1, ], c(xmin, xmax))
  len <- df_funky$total_score * diff(bar)
  g[[1]] <- p +
    annotate("text", x = bar[1] + ifelse(len > 1.3, 0, len) + 0.08, y = rows$y[match(df_funky$id, rows$row_id)],
             label = sprintf("%.2f", df_funky$total_score), hjust = 0, size = 3.4, family = family) +
    annotate("segment", x = mean(c(z[2], col_x("pct_assigned_qced")[1])), y = min(rows$y) - 0.5, yend = max(rows$y) + 0.5,
             colour = "grey60", linewidth = 0.3) +
    annotate("text", x = na$x, y = na$y, label = "×", colour = "grey70", size = 3.5, family = family)
  g$patches$layout$heights[2] <- 0
  legs <- lapply(seq_len(length(g[[3]])), function(i) g[[3]][[i]])
  lw <- sapply(legs, `[[`, "width")
  g[[3]] <- patchwork::wrap_plots(c(legs[1], list(patchwork::plot_spacer()), legs[-1], list(patchwork::plot_spacer())),
                                  nrow = 1, widths = c(lw[1], 0.5, lw[-1], max(g[[1]]$width - sum(lw) - 0.5, 0.01)))
  g <- g & coord_equal(expand = FALSE, clip = "off") & theme(plot.margin = margin(2, 2, 2, 2))

  w <- g$width; h <- g$height
  g <- g & theme(plot.background = element_rect(fill = NA, colour = NA))
  g <- g + patchwork::plot_annotation(theme = theme(plot.background = element_rect(fill = NA, colour = NA)))
  ggsave(paste0(out_file, ".png"), g, width = w, height = h, dpi = 600, bg = "transparent", type = "cairo")
  ggsave(paste0(out_file, ".svg"), g, width = w, height = h, bg = "transparent")
}

In [8]:
for name, table in tables.items():
    for rows, df_funky in [("main", table[table.index.isin(main_methods)]), ("full", table)]:
        df_funky = df_funky.reset_index()
        out_file = str(out_dir / f"{name}_{rows}")
        %R -i df_funky -i out_file -i metric_labels print(plot_benchmark_heatmap(df_funky, out_file, metric_labels))

df_expr = tables["combined"].assign(total_score=lambda df: df.expr_score).sort_values("total_score", ascending=False).reset_index()
out_file = str(out_dir / "combined_expression")
%R -i df_expr -i out_file -i metric_labels print(plot_benchmark_heatmap(df_expr, out_file, metric_labels, score_label = "Expr. score"))

ℹ No row info was provided, assuming all rows in `data` are to be plotted.
ℹ Row info did not contain group information, assuming rows are ungrouped.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
[1] "/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/merged/benchmark_tables/aging_main.svg"



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



ℹ No row info was provided, assuming all rows in `data` are to be plotted.
ℹ Row info did not contain group information, assuming rows are ungrouped.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
[1] "/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/merged/benchmark_tables/aging_full.svg"
ℹ No row info was provided, assuming all rows in `data` are to be plotted.
ℹ Row info did not contain group information, assuming rows are ungrouped.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Coordinate system already present.
ℹ Adding new coordinate system, which 

array(['/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/merged/benchmark_tables/combined_expression.svg'],
      dtype='<U107')

## Sensitivity checks

In [9]:
real = tables["combined"].query("~is_control")
cohort_scores = pd.DataFrame({c: tables[c].total_score for c in cohorts}).reindex(real.index)
mean_rank = cohort_scores.rank(ascending=False).mean(axis=1)

print("shared range and MECR fit:", real.total_score.corr(pooled_score[real.index], method="spearman"))
print("mean rank across cohorts:", real.total_score.corr(-mean_rank, method="spearman"))

shared range and MECR fit: 0.9755808440018965
mean rank across cohorts: 0.9932983550273304


## Missing values

In [10]:
for name, table in tables.items():
    missing = table[metrics].isna()
    missing["ficture_f1_score"] &= ~table.na_ficture
    missing["Ovrlpy_stats_mean_integrity"] &= ~table.na_z
    print(name, {metric: missing.index[missing[metric]].tolist() for metric in metrics if missing[metric].any()})

aging {'marker_f1': ['Baysor Sopa (CP2 T, 0.8)', 'Baysor Sopa (CP2 P, 0.8)', 'Ficture Seg (CP1 3D N)', 'Raster 5μm'], 'ficture_f1_score': ['CellposeSAM Sopa', 'Cellpose1 P+N VPT', 'Baysor Sopa (CP1 P, 0.8)', 'Baysor Sopa (CP1 N, 1.0)', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
ABCAtlas {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor 3D', 'Baysor']}
VizgenMouseBrain {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
combined {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
